### Search Engine With Tools And Agents

In [ ]:
## Arxiv--Research
## Tools creation
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

/var/folders/w7/11s0jcxd0dlf09gcd87g3pyw0000gn/T/ipykernel_46499/554445524.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun


In [2]:
## Used the inbuilt tool of wikipedia
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250) # wikipedia api wrapper is used to fetch the content of the wikipedia page and then it is passed to the wikipedia query run tool which is used to query the content of the wikipedia page.
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [3]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [ ]:
tools=[wiki,arxiv] # combining tools

In [5]:
## Custom tools[RAG Tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

/Users/manashmahanta/Desktop/Udemy Generative AI/Langchainprojects/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [8]:
import os

from dotenv import load_dotenv
load_dotenv()

# os.environ['OPEN_AI_API_KEY']=os.getenv("OPEN_AI_API_KEY")

True

In [ ]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents,OpenAIEmbeddings())
retriever=vectordb.as_retriever()
retriever # retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x129db6290>, search_kwargs={})

In [11]:
#from langchain.tools.retriever import create_retriever_tool
from langchain_core.tools import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith-search","Search any information about Langsmith ")

retriever_tool.name

'langsmith-search'

In [12]:
tools=[wiki,arxiv,retriever_tool]

In [13]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/Users/manashmahanta/Desktop/Udemy Generative AI/Langchainprojects/.venv/lib/python3.10/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about Langsmith ', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x129dc0940>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x129dc09d0>)]

In [14]:
## Run all this tools with Agents and LLM Models

## Tools, LLM-->AgentExecutor
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import openai
load_dotenv()
import os

groq_api_key=os.getenv("GROQ_API_KEY")
openai.api_key=os.getenv("OPENAI_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b")

# Llama3-8b-8192 is decommissioned. 

In [21]:
## Prompt Template
# from langchain_classic import hub
# prompt=hub.pull("hwchase17/openai-functions-agent",dangerously_pull_public_prompt=True)
# prompt.messages # Mostly use chatptompt template and use your own prompt.
# from langchain_classic import hub

# # Add the safety flag inside the hub.pull constructor
# prompt = hub.pull("hwchase17/openai-tools-agent", dangerously_pull_public_prompt=True)

# # Verify the loaded message array structures
# print(prompt.messages)
# 1. Import the native SDK client
# from langchainhub import Client

# # 2. Initialize the client
# client = Client()

# # 3. Pull the prompt using the required safety flag
# prompt = client.pull_prompt("hwchase17/openai-tools-agent", dangerously_pull_public_prompt=True)

# # 4. Verify your template is ready
# print(prompt.messages)
# import os
# from dotenv import load_dotenv
# from langchainhub import Client

# # 1. Load your .env file containing LANGCHAIN_API_KEY
# load_dotenv()

# # 2. Re-verify the environment mapping explicitly if needed
# os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

# # 3. Pull the prompt with an authenticated connection
# client = Client()
# prompt = client.pull_prompt("hwchase17/openai-tools-agent", dangerously_pull_public_prompt=True)
# print(prompt.messages)


from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Build the exact blueprint of "hwchase17/openai-tools-agent" locally
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Verify your template structure matches perfectly
print(prompt.messages)


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant.'), additional_kwargs={}), MessagesPlaceholder(variable_name='chat_history', optional=True), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}), MessagesPlaceholder(variable_name='agent_scratchpad')]


In [23]:
## Agents
from langchain_classic.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm,tools,prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [24]:
## Agent Executer
from langchain_classic.agents import AgentExecutor
agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [25]:
agent_executor.invoke({"input":"Tell me about Langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'LangSmith'}`


LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.Get startedCreate an accountSign up at smith.langchain.c

{'input': 'Tell me about Langsmith',
 'output': '**LangSmith – Observability for LLM‑powered applications**\n\nLangSmith (formerly “LangChain\u202fObservability”) is the monitoring, debugging, and evaluation platform built by the LangChain team for any application that uses large language models (LLMs). It gives developers the same kind of end‑to‑end visibility that traditional observability tools provide for micro‑services, but it’s tuned to the unique workflow of LLM apps—prompt construction, model calls, retrieval‑augmented generation (RAG), tool use, and downstream evaluation.\n\n---\n\n## Core Capabilities\n\n| Capability | What it does | Why it matters |\n|------------|--------------|----------------|\n| **Tracing** | Every LLM request (prompt, model, parameters, response, tool calls, callbacks) is recorded as a *trace*. Traces can be viewed in the UI, filtered, exported, or queried via the API. | Gives you a replayable, searchable log of exactly what the model saw and produced, 

In [26]:
agent_executor.invoke({"input":"What is machine learning"})



> Entering new AgentExecutor chain...
**Machine learning (ML)** is a sub‑field of artificial intelligence (AI) that focuses on developing algorithms and statistical models that enable computers to **learn from data** and make decisions or predictions without being explicitly programmed for each specific task.

---

## Core Idea
- **Learning from Experience:** An ML system improves its performance on a task as it is exposed to more data (the “experience”).
- **Generalization:** The goal is to infer patterns that work not only on the training data but also on new, unseen data.

---

## Typical Workflow

| Step | Description |
|------|-------------|
| **1. Data Collection** | Gather raw data relevant to the problem (images, text, sensor readings, etc.). |
| **2. Data Preparation** | Clean, label, normalize, and split data into training/validation/test sets. |
| **3. Model Selection** | Choose a family of algorithms (e.g., linear models, decision trees, neural networks). |
| **4. Trainin

{'input': 'What is machine learning',
 'output': '**Machine learning (ML)** is a sub‑field of artificial intelligence (AI) that focuses on developing algorithms and statistical models that enable computers to **learn from data** and make decisions or predictions without being explicitly programmed for each specific task.\n\n---\n\n## Core Idea\n- **Learning from Experience:** An ML system improves its performance on a task as it is exposed to more data (the “experience”).\n- **Generalization:** The goal is to infer patterns that work not only on the training data but also on new, unseen data.\n\n---\n\n## Typical Workflow\n\n| Step | Description |\n|------|-------------|\n| **1. Data Collection** | Gather raw data relevant to the problem (images, text, sensor readings, etc.). |\n| **2. Data Preparation** | Clean, label, normalize, and split data into training/validation/test sets. |\n| **3. Model Selection** | Choose a family of algorithms (e.g., linear models, decision trees, neural n

In [28]:
# agent_executor.invoke({"input":"What's the paper 1706.03762 about?"})